In [1]:
import pandas as pd


In [2]:
listings = pd.read_csv('/content/listings.csv.gz')
# reviews = pd.read_csv('/content/reviews.csv')
# neighbours = pd.read_csv('/content/neighbourhoods.csv')


FileNotFoundError: [Errno 2] No such file or directory: '/content/listings.csv.gz'

In [ ]:
df = listings.copy()

In [ ]:
df.shape

In [ ]:
df.head(1)

In [ ]:
df.info()

In [ ]:
df.index

In [ ]:
df.describe()

In [ ]:
 null = df.isnull().sum()
 null_columns  = null[null == df.shape[0]].index
 null_columns




In [ ]:
df = df.drop(columns = null_columns)

In [ ]:
df.shape

## Step: Handling fully-empty columns
Dropped 14 columns that were 100% missing (e.g., host_about, license) —
these fields simply weren't captured in Bangkok's scrape. Not a data
quality issue, just missing by design in this dataset.

In [ ]:
 null_summary =  df.isnull().sum()
 null_summary[null_summary > 0].sort_values(ascending = False)

In [ ]:
df[df['maximum_minimum_nights'].isnull()]['maximum_minimum_nights']


In [ ]:
df.loc[[7948	, 28024]]


In [ ]:
df.loc[[7948	, 28024]].isnull().sum().sort_values(ascending = False).head(10)


In [ ]:
# df['maximum_minimum_nights'].median()
# df['minimum_minimum_nights'].median()
# df['minimum_maximum_nights'].median()
# df['maximum_maximum_nights'].median()




In [ ]:
df['maximum_minimum_nights'].median()
df['minimum_minimum_nights'].median()
df['maximum_maximum_nights'].median()
df['minimum_maximum_nights'].median()
df['maximum_nights'].median()
df['minimum_nights'].median()



In [ ]:
df['maximum_minimum_nights'].mean()
df['minimum_minimum_nights'].mean()
df['maximum_maximum_nights'].mean()
df['minimum_maximum_nights'].mean()
df['maximum_nights'].mean()
df['minimum_nights'].mean()


In [ ]:
cols_to_fill = ['maximum_minimum_nights' , 'minimum_minimum_nights' , 'maximum_maximum_nights' , 'minimum_maximum_nights' , 'maximum_nights' , 'minimum_nights' ]


In [ ]:
for col in cols_to_fill :
  df[col] = df[col].fillna(df[col].median())


In [ ]:
df[cols_to_fill].isnull().sum()

In [ ]:
 null_summary =  df.isnull().sum()
 null_summary[null_summary > 0].sort_values(ascending = False)

In [ ]:
df[df['host_profile_url'].isnull()].isnull().sum()

In [ ]:
df['host_profile_url']

## Step: Near-complete columns (nights group + host_profile_url)

7 columns had only 2 missing rows (7948, 28024) — verified these are real,
active listings, not broken data. Found strong right-skew in the numeric
columns (e.g., max_nights: median=365, mean=553,596), so imputed using median
instead of mean. host_profile_url left as NaN — not relevant to pricing analysis.

In [ ]:
df.head(1)

In [ ]:
 null_summary =  df.isnull().sum()
 null_summary[null_summary > 0].sort_values(ascending = False)

In [ ]:
[col for col in df.columns if 'review' in col.lower()]

In [ ]:
df[df['number_of_reviews'] == 0].shape

In [ ]:
df['has_reviews'] = df['number_of_reviews'] > 0
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)


In [ ]:
df['reviews_per_month'].isnull().sum()

In [ ]:
df.head(1)

In [ ]:
 null_summary =  df.isnull().sum()
 null_summary[null_summary > 0].sort_values(ascending = False)

In [ ]:
df[['bathrooms', 'bedrooms', 'beds']].describe()

In [ ]:
df['room_type'].head()

In [ ]:
df.groupby('room_type')[['bathrooms' , 'bedrooms' , 'beds']].median()

In [ ]:
for col in ['bathrooms', 'bedrooms', 'beds']:
    df[col] = df.groupby('room_type')[col].transform(lambda x: x.fillna(x.median()))

print(df[['bathrooms', 'bedrooms', 'beds']].isnull().sum())


In [ ]:
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending = False)

In [ ]:
df['bathrooms_text']

## Step: bathrooms, bedrooms, beds

Checked median by room_type before deciding fill strategy. Found medians
were nearly identical across room types in this dataset, but chose to
impute using room_type-grouped median anyway (rather than flat overall
median) since it's conceptually more accurate and costs nothing extra.
Used median over mean due to right-skew from outliers (e.g., 50 bathrooms).

In [ ]:
df.head(2)

In [ ]:
null_summary = df.isnull().sum()
null_summary[null_summary > 0].sort_values(ascending = False)

In [ ]:
df[['bathrooms' , 'bedrooms' , 'beds']].isnull().sum()

In [ ]:
df.columns

In [ ]:
df[df['price'].isnull()][['room_type', 'has_availability', 'number_of_reviews', 'availability_365']].describe(include='all')

In [ ]:
df.shape

In [ ]:
df = df.dropna(subset = ['price'])
df.shape

In [ ]:
null_summary = df.isnull().sum()
null_summary[null_summary > 0].sort_values(ascending = False)


## Step: price (target variable) - dropped missing rows

2082 rows missing price. Checked has_availability, reviews, availability_365 —
no clear "inactive listing" pattern found. Price-derived columns
(price_quote_*, estimated_revenue) missing the same rows — likely a scraping
gap, not bad data. Dropped these rows rather than impute, since price is the
target variable and shouldn't be faked.

In [ ]:
null_summary = df.isnull().sum()
null_summary[null_summary > 0].sort_values(ascending = False)

In [ ]:
df = df.dropna(subset = ['has_availability'])

In [ ]:
df['has_about'] = df['host_about'].notna()

In [3]:
df.head(2)

NameError: name 'df' is not defined

In [4]:
null_summary = df.isnull().sum()
null_summary[null_summary > 0].sort_values(ascending = False)

NameError: name 'df' is not defined

## Step: Final missing-value decisions

host_about: engineered has_about boolean flag (missing bio may weakly signal
host trust/quality, worth testing against price later).

host_location, description, bathrooms_text: low relevance to pricing question,
left as NaN — no flag created, avoids adding noise features without a clear reason.

has_availability: only 9 rows missing, dropped them.

Review-score columns remain NaN (already explained earlier — no reviews yet,
not a data error). Missing-value cleaning phase now complete.

In [5]:
df.head(2)

NameError: name 'df' is not defined

In [ ]:
df['price'] = df['price'].replace('[\$,]', '', regex=True).astype(float)

In [ ]:
baseline = df.groupby(['neighbourhood_cleansed' , 'room_type'])['price'].median()

In [6]:
baseline

NameError: name 'baseline' is not defined

In [7]:
baseline.describe()

NameError: name 'baseline' is not defined

## Baseline: Median price by neighbourhood + room_type

Calculated median price grouped by neighbourhood_cleansed + room_type as a
naive baseline (157 groups). Median across groups ≈ ฿1,259, but huge spread:
min ฿299, max ฿11,412 (38x range). Shows location + room type alone explain
massive price variation in Bangkok's market. This becomes the benchmark any
future prediction model would need to beat.

In [8]:
df[['price' , 'bedrooms' , 'bathrooms' , 'beds']].corr()['price']

NameError: name 'df' is not defined

## Hypothesis 1: Bedrooms/bathrooms directly drive price — REJECTED

Correlation with price: bedrooms=0.08, bathrooms=0.08, beds=0.07 — very weak.
Size alone doesn't explain price variation much. Possible reasons: outliers
skewing the relationship, room_type may matter more than raw size, or
relationship isn't linear. Worth testing room_type next.

In [9]:
df.groupby('room_type')['price'].median().sort_values(ascending = False)

NameError: name 'df' is not defined

In [10]:
df.groupby('neighbourhood_cleansed')['price'].median().sort_values(ascending = False)

NameError: name 'df' is not defined

In [ ]:
room_type_std = df.groupby('room_type')['price'].median().std()
neighbourhood_std = df.groupby('neighbourhood_cleansed')['price'].median().std()

In [11]:
room_type_std

NameError: name 'room_type_std' is not defined

In [12]:
neighbourhood_std

NameError: name 'neighbourhood_std' is not defined

## Hypothesis 2: Room type vs neighbourhood — room type wins

Compared spread of median prices: room_type std = 674, neighbourhood std = 414.
Room type creates ~63% more price variation than neighbourhood. Makes sense —
room type (shared vs private vs entire home) is a bigger structural difference
than location alone. Room type appears to be the stronger price driver so far.

In [ ]:
df.head(2)

In [13]:
df.groupby('has_reviews')['price'].median()



NameError: name 'df' is not defined

In [14]:
df['has_reviews'].value_counts()


NameError: name 'df' is not defined

## Hypothesis 3: Reviews affect price — REJECTED

Median price: no reviews = ฿1601, has reviews = ฿1580. Nearly identical,
negligible difference. Having reviews (or not) doesn't meaningfully relate
to price. Hosts don't appear to price new/unreviewed listings differently.

In [15]:
import matplotlib.pyplot as plt
df.boxplot(column = 'price' , by = 'room_type' , figsize = (8,5))
plt.title('Price Distribution by Room Type')
plt.suptitle('')
plt.ylabel('Price (THB)')
plt.show()

NameError: name 'df' is not defined

In [16]:
df[df['price'] > 500000][['name' , 'room_type' , 'neighbourhood_cleansed' , 'price' , 'availability_365']]

NameError: name 'df' is not defined

In [ ]:
df = df[df['price'] < 500000]

In [ ]:
df.shape

## Outlier discovery: extreme price listings

Found 7 listings priced above ฿900,000/night — implausible for any Bangkok
property. 3 had names explicitly indicating "not for sale/room swap"
(Chinese text). Other 4 had normal-looking names but equally implausible
prices — likely data entry errors or the same unbookable-pricing pattern
without a naming clue. Removed all 7 (price < 500000) as non-market prices.

In [17]:
df.boxplot(column = 'price' , by = 'room_type' , figsize = (8,5))
plt.title('Price Distribution by Room Type')
plt.suptitle('')
plt.ylabel('Price (THB)')
plt.show()

NameError: name 'df' is not defined

In [18]:
import matplotlib.pyplot as plt

df.boxplot(column='price', by='room_type', figsize=(8,5))
plt.yscale('log')
plt.title('Price Distribution by Room Type (log scale)')
plt.suptitle('')
plt.ylabel('Price (THB, log scale)')
plt.show()

NameError: name 'df' is not defined

## Room type - refined finding

Log-scale boxplot reveals: Entire home/apt, Hotel room, and Private room have
similar median prices (฿1500-2000) with heavy overlap. Shared room is the
clear outlier — notably cheaper (฿400-500 median). The earlier "room_type
std" finding was mostly driven by Shared room being different, not a smooth
gradient across all 4 types. All categories (except Shared room) have long
tails of legitimate high-end listings (villas, premium condos) — kept in the
dataset as real market data, not errors.

In [ ]:
#gbvcsd